
<h1 id="Analysis:-Sentiment-Analysis-of-Working-Capital-Management-Efficiency-of-Nifty-500-on-the-Firm-Performance-keeping-BRSR-as-Moderator,Pre-and-Post-4years">Analysis: Sentiment Analysis of Working Capital Management Efficiency of Nifty 500 on the Firm Performance keeping BRSR as Moderator,Pre and Post 4years</h1><p>Moderating role of BRSR disclosure intensity and sentiment, 2019–2026</p>
<p>This notebook runs the full analysis plan: data cleaning, transformation, exploration, fixed-effects models, BRSR moderation, optimal working capital, pre/post-mandate tests, COVID shock test, endogeneity checks and robustness.</p>
<p>Packages used: pandas, numpy, matplotlib, scipy, re, difflib (plus openpyxl, which pandas needs to read Excel).</p>



<h2 id="1.-Loading-the-libraries">1. Loading the libraries</h2>


In [1]:
import re, difflib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import skimpy
from scipy import stats

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)
plt.rcParams.update({'figure.figsize': (8, 4.5), 'axes.grid': True, 'grid.alpha': 0.3})



<h2 id="2.-Loading-and-reshaping-the-data">2. Loading and reshaping the data</h2>



<h3 id="2.1-Financial-data">2.1 Financial data</h3>


In [2]:
raw = pd.read_excel('Final_Data_Minor_project.xlsx', header=None)
years = raw.iloc[0, 1:].astype(int)          # row 0 holds the year
varnames = raw.iloc[1, 1:]                   # row 1 holds the variable name
body = raw.iloc[2:].set_index(0).apply(pd.to_numeric, errors='coerce')
body.columns = pd.MultiIndex.from_arrays([varnames.values, years.values], names=['var', 'year'])

fin = (body.T.reset_index().melt(id_vars=['var', 'year'], var_name='firm')
       .pivot_table(index=['firm', 'year'], columns='var', values='value', aggfunc='first')
       .reset_index())
fin.columns.name = None
fin = fin.rename(columns={'TobinsQ': 'Q', 'PBRatio': 'PB', 'PromoterHoldings': 'Promoter',
                          'RnDIntensity(%)': 'RnD', 'Inflation (CPI)': 'Infl', 'GDPRate (%)': 'GDP'})
print(fin.shape, '| firms:', fin.firm.nunique(), '| years:', fin.year.min(), '-', fin.year.max())
fin.head()


(3992, 18) | firms: 499 | years: 2019 - 2026


,firm,year,BoardInd,BoardSize,CovidDummy,FirmAge,FirmSize,GDP,Infl,Leverage,PB,Promoter,ROA,ROE,RnD,Tangibility,Q,lnCCC
0,360 One Wam Ltd.,2019,0.111111,9.0,0.0,11.0,7.889957,3.871437,3.729506,0.155039,NaN,64.247,0.056839,2.2781,0.0,0.0,0.155039,NaN
1,360 One Wam Ltd.,2020,0.363636,11.0,1.0,12.0,7.833196,-5.777725,6.623437,0.000000,3.769791,22.930,0.076687,11.0068,0.0,0.0,3.475521,NaN
2,360 One Wam Ltd.,2021,0.363636,11.0,0.0,13.0,7.903925,9.689592,5.131407,0.086315,4.482887,22.920,0.236483,26.9213,0.0,0.0,4.108705,NaN
3,360 One Wam Ltd.,2022,0.363636,11.0,0.0,14.0,8.325415,7.609365,6.699034,0.180233,6.106735,23.140,0.098712,16.7103,0.0,0.0,3.767059,NaN
4,360 One Wam Ltd.,2023,0.307692,13.0,0.0,15.0,8.306917,7.210225,5.649143,0.303694,6.516225,22.020,0.122378,20.5253,0.0,0.0,4.088871,NaN



<p><strong>Interpretation.</strong> The financial file holds 499 firms over 2019–2026 (3,992 firm-years) and 16 variables. Reshaping from wide to long gives one row per firm-year, which is the format every panel method needs.</p>



<h3 id="2.2-BRSR-scores">2.2 BRSR scores</h3>


In [3]:
rawb = pd.read_excel('BRSR_Scores.xlsx', header=None)
yb = rawb.iloc[0, 1:].astype(int)
vb = rawb.iloc[1, 1:]
bodyb = rawb.iloc[2:].set_index(0).apply(pd.to_numeric, errors='coerce')
bodyb.columns = pd.MultiIndex.from_arrays([vb.values, yb.values], names=['var', 'year'])

brsr = (bodyb.T.reset_index().melt(id_vars=['var', 'year'], var_name='firm')
        .pivot_table(index=['firm', 'year'], columns='var', values='value', aggfunc='first')
        .reset_index())
brsr.columns.name = None
brsr = brsr.rename(columns={'ESGDI': 'esgdi', 'ESG-NSI': 'nsi', 'Overall Narrative Sentiment': 'overall_sent',
                            'Net Sentiment Index': 'net_sent', 'Environmental Disclosure': 'env',
                            'Social Disclosure': 'soc', 'Governance Disclosure': 'gov'})
print(brsr.shape, '| firms:', brsr.firm.nunique(), '| years:', sorted(brsr.year.unique()))
brsr.head()


(1607, 9) | firms: 488 | years: [np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]


,firm,year,nsi,esgdi,env,gov,net_sent,overall_sent,soc
0,360 One Wam Ltd.,2023,0.1039,0.5124,0.1690,0.1250,0.0714,0.0750,0.3516
1,360 One Wam Ltd.,2024,0.0973,0.5021,0.1614,0.1255,0.0662,0.0651,0.3366
2,360 One Wam Ltd.,2025,0.1307,0.5089,0.1756,0.1272,0.0936,0.0962,0.3375
3,360 One Wam Ltd.,2026,0.0985,0.5054,0.1943,0.1305,0.0716,0.0757,0.3180
4,A B B India Ltd.,2024,0.1791,0.4688,0.2344,0.1484,0.0938,0.1035,0.3438



<p><strong>Interpretation.</strong> The BRSR file covers 488 firms and four years only (2023–2026), 1,607 firm-years in total.</p>



<h2 id="3.-Merging-the-two-datasets">3. Merging the two datasets</h2><p>Company names differ between the files (for example <code>Bajaj Holdings &amp; Invst. Ltd.</code> vs <code>Bajaj_Holdings_and_Investment</code>). We standardise the names, match exactly, then use <code>difflib</code> for near-matches (cutoff 0.85, so only very close names are joined).</p>


In [4]:
drop_words = r'\b(ltd|limited|pvt|private|india|company|co|corporation|corp|corpn|inc)\b'
for d in (fin, brsr):
    d['key'] = (d.firm.str.lower().str.replace('_', ' ').str.replace('&', 'and')
                .str.replace(drop_words, '', regex=True).str.replace(r'[^a-z0-9]', '', regex=True))

only_b = sorted(set(brsr.key) - set(fin.key))
only_f = sorted(set(fin.key) - set(brsr.key))
near = {k: difflib.get_close_matches(k, only_f, n=1, cutoff=0.85) for k in only_b}
near = {k: v[0] for k, v in near.items() if v}
print('Near-matches accepted:', near)
brsr['key'] = brsr.key.replace(near)

panel = fin.merge(brsr.drop(columns='firm'), on=['key', 'year'], how='left')
panel = panel.sort_values(['firm', 'year']).reset_index(drop=True)
print('BRSR firms matched to financial file:', brsr.key.isin(fin.key).groupby(brsr.firm).any().sum(), 'of', brsr.firm.nunique())
print('Panel shape:', panel.shape)


Near-matches accepted: {'bajajholdingsandinvestment': 'bajajholdingsandinvst', 'pcblcemical': 'pcblchemical'}
BRSR firms matched to financial file: 462 of 488
Panel shape: (3992, 26)



<p><strong>Interpretation.</strong> 462 of the 488 BRSR firms match a firm in the financial file. Only two names needed fuzzy matching (Bajaj Holdings and PCBL), and both are clearly the same companies. The other 26 BRSR firms (for example Pfizer, Blue Dart, Saregama) do not appear in the financial file at all, so they cannot be used.</p>



<h2 id="4.-Data-quality-checks">4. Data quality checks</h2>



<h3 id="Data-exploration">Data exploration</h3>


In [8]:
from skimpy import skim 
skim(panel)


╭──────────────────────────────────────────────── skimpy summary ─────────────────────────────────────────────────╮
│ Data Summary Data Types │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓ ┏━━━━━━━━━━━━━┳━━━━━━━┓ │
│ ┃ Dataframe ┃ Values ┃ ┃ Column Type ┃ Count ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩ ┡━━━━━━━━━━━━━╇━━━━━━━┩ │
│ │ Number of rows │ 3992 │ │ float64 │ 23 │ │
│ │ Number of columns │ 26 │ │ string │ 2 │ │
│ └───────────────────┴────────┘ │ int64 │ 1 │ │
│ └─────────────┴───────┘ │
│ number │
│ ┏━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┓ │
│ ┃ column ┃ NA ┃ NA % ┃ mean ┃ sd ┃ p0 ┃ p25 ┃ p50 ┃ p75 ┃ p100 ┃ hist ┃ │
│ ┡━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━┩ │
│ │ year │ 0 │ 0 │ 2022 │ 2.292 │ 2019 │ 2021 │ 2022 │ 2024 │ 2026 │ ▇▅▃▃▅▇ │ │
│ │ BoardInd │ 70 │ 1.7535070 │ 0.4559 │ 0.1878 │ 0 │ 0.4 │ 0.5 │ 0.5556 │ 1 │ ▂▁▃▇▁ │ │
│ │ │ │ 140280562 │ │ │ │ │ │ │ │ │ │
│ │ BoardSize │ 69 │ 1.7284569 │ 9.174 │ 2.901 │ 0 │ 7 │ 9 │ 11 │ 25 │ ▁▆▇▂ │ │
│ │ │ │ 138276553 │ │ │ │ │ │ │ │ │ │
│ │ CovidDummy │ 0 │ 0 │ 0.125 │ 0.3308 │ 0 │ 0 │ 0 │ 0 │ 1 │ ▇ ▁ │ │
│ │ FirmAge │ 0 │ 0 │ 39.39 │ 25.65 │ -5 │ 23 │ 33 │ 51 │ 132 │ ▃▇▃▂▁ │ │
│ │ FirmSize │ 130 │ 3.2565130 │ 9.13 │ 1.905 │ -4.605 │ 7.999 │ 8.943 │ 10.05 │ 15.85 │ ▇▇▁ │ │
│ │ │ │ 26052104 │ │ │ │ │ │ │ │ │ │
│ │ GDP │ 499 │ 12.5 │ 5.324 │ 4.802 │ -5.778 │ 3.871 │ 7.21 │ 7.609 │ 9.69 │ ▂ ▂▂▇ │ │
│ │ Infl │ 499 │ 12.5 │ 5.026 │ 1.431 │ 2.399 │ 3.73 │ 5.131 │ 6.623 │ 6.699 │ ▃▃ ▇▃▇ │ │
│ │ Leverage │ 614 │ 15.380761 │ 0.1775 │ 0.584 │ 0 │ 0 │ 0.06495 │ 0.234 │ 15.62 │ ▇ │ │
│ │ │ │ 523046091 │ │ │ │ │ │ │ │ │ │
│ │ PB │ 720 │ 18.036072 │ 7.183 │ 23.37 │ 0.03981 │ 1.936 │ 3.939 │ 7.892 │ 965.9 │ ▇ │ │
│ │ │ │ 14428858 │ │ │ │ │ │ │ │ │ │
│ │ Promoter │ 230 │ 5.7615230 │ 58.14 │ 21.51 │ 0 │ 45.62 │ 58.5 │ 72.82 │ 100 │ ▁▂▅▇▆▃ │ │
│ │ │ │ 46092185 │ │ │ │ │ │ │ │ │ │
│ │ ROA │ 130 │ 3.2565130 │ 0.06459 │ 0.1684 │ -4.394 │ 0.02018 │ 0.05824 │ 0.1123 │ 1.997 │ ▇ │ │
│ │ │ │ 26052104 │ │ │ │ │ │ │ │ │ │
│ │ ROE │ 211 │ 5.2855711 │ 13.31 │ 39.78 │ -1071 │ 6.396 │ 13.34 │ 20.3 │ 1063 │ ▁▇ │ │
│ │ │ │ 42284569 │ │ │ │ │ │ │ │ │ │
│ │ RnD │ 146 │ 3.6573146 │ 1.531 │ 40.2 │ 0 │ 0 │ 0 │ 0.412 │ 2476 │ ▇ │ │
│ │ │ │ 292585172 │ │ │ │ │ │ │ │ │ │
│ │ Tangibilit │ 130 │ 3.2565130 │ 0.4854 │ 0.7748 │ 0 │ 0.201 │ 0.4659 │ 0.6739 │ 28.51 │ ▇ │ │
│ │ y │ │ 26052104 │ │ │ │ │ │ │ │ │ │
│ │ Q │ 130 │ 3.2565130 │ 3.146 │ 4.705 │ 0 │ 0.6204 │ 1.769 │ 4.116 │ 99.72 │ ▇ │ │
│ │ │ │ 26052104 │ │ │ │ │ │ │ │ │ │
│ │ lnCCC │ 1227 │ 30.736472 │ -1.67 │ 4.387 │ -13.86 │ -4.863 │ -3.804 │ 3.01 │ 12.86 │ ▅▇▃▃ │ │
│ │ │ │ 945891784 │ │ │ │ │ │ │ │ │ │
│ │ nsi │ 2459 │ 61.598196 │ 0.1391 │ 0.05798 │ -0.0102 │ 0.0976 │ 0.1353 │ 0.1746 │ 0.3708 │ ▁▇▇▃▁ │ │
│ │ │ │ 392785574 │ │ │ │ │ │ │ │ │ │
│ │ esgdi │ 2459 │ 61.598196 │ 0.5078 │ 0.04697 │ 0.125 │ 0.4782 │ 0.509 │ 0.5398 │ 0.6598 │ ▃▇▁ │ │
│ │ │ │ 392785574 │ │ │ │ │ │ │ │ │ │
│ │ env │ 2459 │ 61.598196 │ 0.2151 │ 0.04149 │ 0.1016 │ 0.1851 │ 0.2145 │ 0.243 │ 0.5556 │ ▂▇▂ │ │
│ │ │ │ 392785574 │ │ │ │ │ │ │ │ │ │
│ │ gov │ 2459 │ 61.598196 │ 0.1033 │ 0.02616 │ 0 │ 0.087 │ 0.1022 │ 0.1188 │ 0.255 │ ▃▇▂ │ │
│ │ │ │ 392785574 │ │ │ │ │ │ │ │ │ │
│ │ net_sent │ 2459 │ 61.598196 │ 0.08908 │ 0.04219 │ -0.0105 │ 0.0579 │ 0.0855 │ 0.1151 │ 0.3269 │ ▂▇▅▁ │ │
│ │ │ │ 392785574 │ │ │ │ │ │ │ │ │ │
│ │ overall_se │ 2459 │ 61.598196 │ 0.09016 │ 0.04181 │ -0.0055 │ 0.0598 │ 0.0872 │ 0.1159 │ 0.2994 │ ▂▇▆▁ │ │
│ │ nt │ │ 392785574 │ │ │ │ │ │ │ │ │ │
│ │ soc │ 2459 │ 61.598196 │ 0.3281 │ 0.03973 │ 0.125 │ 0.3025 │ 0.3291 │ 0.3537 │ 0.5556 │ ▁▇▆ │ │
│ │ │ │ 392785574 │ │ │ │ │ │ │ │ │ │
│ └────────────┴──────┴───────────┴─────────┴─────────┴─────────┴─────────┴─────────┴────────┴────────┴────────┘ │
│ string │
│ ┏━━━━━━━━┳━━━━┳━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┓ │
│ ┃ ┃ ┃ ┃ ┃ ┃ ┃ ┃ chars per ┃ word


<h3 id="4.1-Missing-values-by-variable-and-year">4.1 Missing values by variable and year</h3>


In [9]:
(panel.drop(columns=['firm', 'key']).isna().groupby(panel.year).mean().T * 100).round(0)


year,2019,2020,2021,2022,2023,2024,2025,2026
year,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
BoardInd,3.0,3.0,2.0,2.0,2.0,1.0,0.0,1.0
BoardSize,3.0,3.0,2.0,2.0,2.0,1.0,0.0,1.0
CovidDummy,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
FirmAge,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
FirmSize,5.0,4.0,4.0,3.0,3.0,2.0,2.0,3.0
GDP,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0
Infl,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0
Leverage,5.0,4.0,4.0,3.0,3.0,2.0,2.0,100.0
PB,30.0,27.0,24.0,20.0,17.0,14.0,9.0,3.0



<p><strong>Interpretation.</strong> Four things stand out.</p>
<p>(1) Leverage, inflation and GDP are empty for 2026, so 2026 needs special handling.</p>
<p>(2) <code>lnCCC</code> is missing for 29–36% of firm-years in every year, a much larger gap than any other variable.</p>
<p>(3) P/B is missing for up to 30% of firm-years, which is why it is not in the main models.</p>
<p>(4) BRSR columns are empty for 2019–2022 by design, and coverage rises from 52% of firms in 2023 to 89% in 2026.</p>



<h3 id="4.2-Which-firms-have-no-usable-cash-conversion-cycle?">4.2 Which firms have no usable cash conversion cycle?</h3><p><code>lnCCC</code> is the only working-capital variable supplied, and a logarithm is undefined for zero or negative CCC. We check how many firms are affected.</p>


In [10]:
n_obs = panel.groupby('firm').lnCCC.count()
print('Number of years with lnCCC, by firm:')
print(n_obs.value_counts().sort_index().to_string())

has = panel.firm.map(n_obs > 0)
cmp = panel.groupby(has)[['FirmSize', 'ROA', 'ROE', 'Q', 'Leverage']].mean().round(3)
cmp.index = ['No lnCCC in any year', 'Has lnCCC']
cmp


Number of years with lnCCC, by firm:
lnCCC
0    129
1      7
2      4
3      5
4     11
5      6
6      6
7     23
8    308


,FirmSize,ROA,ROE,Q,Leverage
No lnCCC in any year,9.923,0.038,9.779,2.847,0.303
Has lnCCC,8.874,0.073,14.457,3.243,0.137



<p><strong>Interpretation.</strong> 129 of 499 firms (26%) have no valid <code>lnCCC</code> in any year. They include banks, NBFCs and insurers (where CCC has no meaning) but also IT services, hospitals, hotels and telecom firms, where CCC is probably zero or negative so the logarithm cannot be taken. These firms are larger (log assets 9.9 against 8.9), earn less (ROA 3.8% against 7.3%) and are more leveraged (0.30 against 0.14). <strong>The results therefore describe firms with a positive cash conversion cycle, not all of Nifty 500.</strong></p>



<h3 id="4.3-Impossible-or-suspicious-values">4.3 Impossible or suspicious values</h3>


In [11]:
checks = {'FirmAge < 0': (panel.FirmAge < 0).sum(),
          'FirmSize < 0 (ln of assets)': (panel.FirmSize < 0).sum(),
          'Tangibility > 1': (panel.Tangibility > 1).sum(),
          'BoardSize < 3': (panel.BoardSize < 3).sum(),
          'Leverage > 5': (panel.Leverage > 5).sum(),
          'Promoter holding > 100': (panel.Promoter > 100).sum(),
          'R&D intensity = 0 (not reported or none)': (panel.RnD == 0).sum()}
pd.Series(checks, name='count')


FirmAge < 0                                   34
FirmSize < 0 (ln of assets)                   12
Tangibility > 1                               47
BoardSize < 3                                 86
Leverage > 5                                   7
Promoter holding > 100                         0
R&D intensity = 0 (not reported or none)    2157
Name: count, dtype: int64


<p><strong>Interpretation.</strong> Several impossible values exist: negative firm age (34), negative log assets (12), tangibility above 1 (47) and boards of fewer than three directors (86). They are set to missing in the next section. A zero R&amp;D figure appears in 2,157 firm-years; that cannot be told apart from 'not reported', so a reporting indicator is added later.</p>



<h3 id="4.4-Shape-of-the-working-capital-variable">4.4 Shape of the working-capital variable</h3><p>A histogram of <code>lnCCC</code> shows whether the variable is well behaved. We also check whether firms move between regions of the distribution over time.</p>


In [12]:
x = panel.lnCCC.dropna()
plt.hist(x, bins=45, edgecolor='white')
plt.axvline(0, color='red', ls='--'); plt.xlabel('lnCCC'); plt.ylabel('firm-years'); plt.title('Distribution of lnCCC (all firms)')
plt.show()

upper = (panel.lnCCC > 0).where(panel.lnCCC.notna())            # True when exp(lnCCC) > 1
firm_share = upper.groupby(panel.firm).mean().dropna()
print('Share of firm-years with lnCCC > 0:', round(upper.mean(), 3))
print('Firms always below 0:', (firm_share == 0).sum(), '| always above 0:', (firm_share == 1).sum(),
      '| switching between the two:', ((firm_share > 0) & (firm_share < 1)).sum())
panel[panel.firm.str.contains('Reliance Industries|Hindustan Unilever|Asian Paints|D L F')].pivot(index='firm', columns='year', values='lnCCC').round(2)


Share of firm-years with lnCCC > 0: 0.328
Firms always below 0: 177 | always above 0: 47 | switching between the two: 146


year,2019,2020,2021,2022,2023,2024,2025,2026
firm,,,,,,,,
Asian Paints Ltd.,-4.35,-4.38,-4.55,-4.10,-4.27,-4.26,-4.22,-4.48
D L F Ltd.,NaN,8.34,8.46,8.39,8.35,8.28,8.07,8.01
Hindustan Unilever Ltd.,-5.46,-5.52,-5.40,-5.32,-5.13,-5.26,-6.09,-6.23
Reliance Industries Ltd.,-4.89,-4.80,-5.10,-4.93,-4.83,-4.90,-5.94,-5.71



<p><strong>Interpretation.</strong> <code>lnCCC</code> has <strong>two separate clusters</strong>: one centred near −4.5 and one near +4, with few observations around 0. About 33% of firm-years sit above zero. Of firms observed in several years, 177 stay in the lower cluster, 47 stay in the upper and 146 move between them. Examples: DLF is around +8 (inventory-heavy real estate, which fits long cycles), while Reliance, Hindustan Unilever and Asian Paints sit around −4 to −6, which corresponds to cycles of a fraction of a day if the unit is days. This pattern usually means that the lower cluster contains very small positive CCC values (where the logarithm explodes) or a different unit. Hence, every key model is also run on a scale-free rank of lnCCC and on each cluster separately.</p>



<h2 id="5.-Data-cleaning">5. Data cleaning</h2><p>Decisions, each tied to a check above:</p>
<ol>
<li>Keep only firms with at least one valid <code>lnCCC</code> (CCC is undefined for the rest, including banks and NBFCs).</li>
<li>Set impossible values to missing.</li>
<li>Put ROA in percent so it has the same unit as ROE.</li>
<li>2026 leverage is entirely missing in the file; carry forward the 2025 value (flagged, and tested in robustness).</li>
<li>Winsorise continuous variables at the 1st and 99th percentiles.</li>
</ol>


In [13]:
sample_firms = n_obs[n_obs > 0].index
df = panel[panel.firm.isin(sample_firms)].copy()

df.loc[df.FirmAge < 0, 'FirmAge'] = np.nan
df.loc[df.FirmSize < 0, 'FirmSize'] = np.nan
df.loc[df.Tangibility > 1, 'Tangibility'] = np.nan
df.loc[df.BoardSize < 3, 'BoardSize'] = np.nan
df['ROA'] = df.ROA * 100

prev_lev = df.groupby('firm').Leverage.shift(1)
df['lev_filled'] = ((df.year == 2026) & df.Leverage.isna() & prev_lev.notna()).astype(int)
df.loc[df.lev_filled == 1, 'Leverage'] = prev_lev

cont = ['lnCCC', 'ROA', 'ROE', 'Q', 'FirmSize', 'Leverage', 'Tangibility', 'PB', 'RnD', 'Promoter',
        'esgdi', 'nsi', 'net_sent', 'overall_sent', 'env', 'soc', 'gov']      # BRSR scores are winsorised too
low, high = df[cont].quantile(0.01), df[cont].quantile(0.99)
df[cont] = df[cont].clip(low, high, axis=1)
print('Firms kept:', df.firm.nunique(), '| firm-years:', len(df), '| 2026 leverage values carried forward:', df.lev_filled.sum())


Firms kept: 370 | firm-years: 2960 | 2026 leverage values carried forward: 369



<p><strong>Interpretation.</strong> 370 firms and 2,960 firm-years remain after dropping firms with no usable <code>lnCCC</code>. 2026 leverage was carried forward from 2025 for 369 firms (it is tested in robustness by dropping 2026).</p>



<h2 id="6.-Data-transformation">6. Data transformation</h2>


In [14]:
df['Post'] = (df.year >= 2023).astype(int)                 # post-mandate period: 2023-2026
df['lnAge'] = np.log1p(df.FirmAge)
df['RnD_rep'] = (df.RnD > 0).astype(int)                   # R&D reported indicator
df['lnCCC_c'] = df.lnCCC - df.lnCCC.mean()                 # centred WCM for interactions
df['lnCCC_sq'] = df.lnCCC_c ** 2
df['has_brsr'] = df.esgdi.notna().astype(int)

# standardised BRSR measures (mean 0, SD 1) -> coefficients read "per one SD"
for v in ['esgdi', 'nsi', 'net_sent', 'env', 'soc', 'gov']:
    df[v + '_z'] = (df[v] - df[v].mean()) / df[v].std()

# firm-level (time-invariant) BRSR exposure: average over available years
for v in ['esgdi', 'nsi']:
    df[v + '_avg'] = df.groupby('firm')[v].transform('mean')
    df[v + '_avg_z'] = (df[v + '_avg'] - df[v + '_avg'].mean()) / df[v + '_avg'].std()

# lags and changes
df['lnCCC_lag'] = df.groupby('firm').lnCCC.shift(1)
df['ROA_lag'] = df.groupby('firm').ROA.shift(1)
df['d_lnCCC'] = df.lnCCC - df.lnCCC_lag
print(df[['lnCCC', 'lnCCC_c', 'esgdi', 'esgdi_z', 'Post', 'has_brsr']].describe().round(3).T)


           count   mean    std    min    25%    50%    75%    max
lnCCC     2765.0 -1.679  4.302 -8.335 -4.863 -3.804  3.010  7.857
lnCCC_c   2765.0 -0.000  4.302 -6.657 -3.185 -2.125  4.689  9.536
esgdi     1136.0  0.514  0.044  0.400  0.484  0.515  0.545  0.622
esgdi_z   1136.0  0.000  1.000 -2.578 -0.677  0.027  0.706  2.453
Post      2960.0  0.500  0.500  0.000  0.000  0.500  1.000  1.000
has_brsr  2960.0  0.384  0.486  0.000  0.000  0.000  1.000  1.000



<p><strong>Interpretation.</strong> Centred lnCCC has mean 0 as intended. ESGDI and the other BRSR scores are now standardised (mean 0, SD 1) and exist only for 2023–2026 (1,136 firm-years). <code>Post</code> splits the panel evenly into 2019–2022 and 2023–2026.</p>



<h2 id="7.-Sample-construction">7. Sample construction</h2><p><strong>Sample A</strong> is the full 2019–2026 panel used for the baseline, non-linearity, COVID and pre/post tests. <strong>Sample B</strong> is the post-mandate panel (2023–2026) where BRSR scores exist; it is the only sample used for moderation. BRSR scores are never filled for 2019–2022.</p>


In [15]:
A = df.dropna(subset=['lnCCC', 'ROA'])
B = df[(df.Post == 1)].dropna(subset=['lnCCC', 'ROA', 'esgdi', 'nsi'])

flow = pd.DataFrame({
    'Stage': ['Financial file (all firms)', 'Firms with at least one valid lnCCC',
              'Sample A: firm-years with lnCCC and ROA', 'Sample B: Sample A, 2023-26, with BRSR scores'],
    'Firms': [panel.firm.nunique(), df.firm.nunique(), A.firm.nunique(), B.firm.nunique()],
    'Firm-years': [len(panel), len(df), len(A), len(B)]})
flow


,Stage,Firms,Firm-years
0,Financial file (all firms),499,3992
1,Firms with at least one valid lnCCC,370,2960
2,Sample A: firm-years with lnCCC and ROA,370,2764
3,"Sample B: Sample A, 2023-26, with BRSR scores",330,1087



<p><strong>Interpretation.</strong> Sample A (all years) has 2,764 firm-years for 370 firms. Sample B (2023–2026 with BRSR scores) has 1,087 firm-years for 330 firms. Regression samples are somewhat smaller because control variables are missing for some firms.</p>



<h2 id="8.-Exploratory-data-analysis">8. Exploratory data analysis</h2>



<h3 id="8.1-Descriptive-statistics">8.1 Descriptive statistics</h3>


In [16]:
main = ['lnCCC', 'ROA', 'ROE', 'Q', 'FirmSize', 'Leverage', 'Tangibility', 'Promoter', 'BoardInd', 'BoardSize', 'RnD']
A[main].describe(percentiles=[0.25, 0.5, 0.75]).T.round(3)


,count,mean,std,min,25%,50%,75%,max
lnCCC,2764.0,-1.682,4.299,-8.335,-4.865,-3.805,3.009,7.857
ROA,2764.0,7.633,7.899,-25.665,3.142,6.756,11.475,35.231
ROE,2723.0,14.522,16.047,-59.549,7.048,13.621,20.559,77.920
Q,2764.0,3.177,3.485,0.000,0.813,2.065,4.303,19.492
FirmSize,2764.0,8.942,1.418,5.382,8.021,8.861,9.771,12.756
Leverage,2764.0,0.131,0.146,0.000,0.001,0.078,0.220,0.643
Tangibility,2722.0,0.505,0.235,0.000,0.320,0.522,0.681,0.958
Promoter,2694.0,58.725,18.522,6.823,46.860,58.625,71.128,100.000
BoardInd,2755.0,0.459,0.174,0.000,0.400,0.500,0.556,1.000
BoardSize,2718.0,9.399,2.747,3.000,8.000,9.000,11.000,22.000



<p><strong>Interpretation.</strong> The average firm earns an ROA of 7.6% and an ROE of 14.5%, with a median Tobin's Q of 2.1. Leverage is low (mean 0.13 of assets). Most firms report no R&amp;D (median 0.07%).</p>


In [17]:
B[['esgdi', 'nsi', 'net_sent', 'overall_sent', 'env', 'soc', 'gov']].describe().T.round(3)


,count,mean,std,min,25%,50%,75%,max
esgdi,1087.0,0.514,0.044,0.400,0.484,0.515,0.545,0.622
nsi,1087.0,0.139,0.058,0.018,0.098,0.136,0.174,0.296
net_sent,1087.0,0.088,0.042,0.007,0.057,0.085,0.113,0.207
overall_sent,1087.0,0.089,0.042,0.008,0.059,0.086,0.114,0.208
env,1087.0,0.222,0.038,0.133,0.196,0.223,0.249,0.316
soc,1087.0,0.329,0.039,0.232,0.303,0.331,0.357,0.423
gov,1087.0,0.100,0.024,0.050,0.083,0.099,0.114,0.171



<p><strong>Interpretation.</strong> On average 51% of scored BRSR sentences are about ESG topics (ESGDI), and the average tone is mildly positive (ESG-NSI of 0.14). Both vary little across firms (SD about 0.04–0.06), which limits how much they can move a regression slope. Social disclosure (33%) is the largest pillar and governance (10%) the smallest.</p>



<h3 id="8.2-Pre--versus-post-mandate-comparison">8.2 Pre- versus post-mandate comparison</h3><p>These are descriptive comparisons (repeated observations of the same firms), not substitutes for the panel regressions.</p>


In [18]:
rows = []
for v in ['lnCCC', 'ROA', 'ROE', 'Q']:
    pre, post = A.loc[A.Post == 0, v].dropna(), A.loc[A.Post == 1, v].dropna()
    rows.append({'variable': v, 'mean_pre': pre.mean(), 'mean_post': post.mean(),
                 'median_pre': pre.median(), 'median_post': post.median(),
                 't_test_p': stats.ttest_ind(pre, post, equal_var=False).pvalue,
                 'mannwhitney_p': stats.mannwhitneyu(pre, post).pvalue})
pd.DataFrame(rows).round(4)


,variable,mean_pre,mean_post,median_pre,median_post,t_test_p,mannwhitney_p
0,lnCCC,-1.4597,-1.8958,-3.6714,-3.9495,0.0076,0.0002
1,ROA,6.9431,8.2949,5.9937,7.4800,0.0000,0.0000
2,ROE,13.5442,15.4554,13.1488,14.0779,0.0019,0.0042
3,Q,2.4648,3.8614,1.3436,2.7167,0.0000,0.0000



<p><strong>Interpretation.</strong> Compared with 2019–2022, the 2023–2026 period has shorter cycles (median lnCCC −3.95 against −3.67), higher ROA (8.3% against 6.9%), higher ROE (15.5% against 13.5%) and higher Tobin's Q (3.9 against 2.5). All differences are significant at 1%. This is a descriptive comparison only: the pre period contains the COVID shock and the post period contains the recovery, so it does not show a BRSR effect.</p>



<h3 id="8.3-Trends-over-time">8.3 Trends over time</h3>


In [19]:
fig, ax = plt.subplots(2, 2, figsize=(11, 7))
for a, v in zip(ax.flat, ['lnCCC', 'ROA', 'ROE', 'Q']):
    g = A.groupby('year')[v]
    a.plot(g.mean().index, g.mean(), marker='o', label='mean')
    a.plot(g.median().index, g.median(), marker='s', ls='--', label='median')
    a.axvline(2022.5, color='grey', ls=':')            # BRSR mandate
    a.set_title(v); a.legend()
plt.suptitle('Yearly trends (dotted line = start of BRSR mandate period)')
plt.tight_layout(); plt.show()



<p><strong>Interpretation.</strong> ROA, ROE and Q all fall in 2020–21 and recover from 2022, matching the pandemic and rebound. Median lnCCC drifts downward (shorter cycles) over the period. Nothing changes abruptly at the dotted line, which is a first sign that the mandate year itself is not a clear break.</p>



<h3 id="8.4-Distribution-of-the-BRSR-measures">8.4 Distribution of the BRSR measures</h3>


In [20]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, v, t in zip(ax, ['esgdi', 'nsi'], ['ESG disclosure intensity (ESGDI)', 'ESG narrative sentiment (ESG-NSI)']):
    a.boxplot([B.loc[B.year == y, v] for y in [2023, 2024, 2025, 2026]])
    a.set_xticklabels([2023, 2024, 2025, 2026]); a.set_title(t)
plt.tight_layout(); plt.show()



<p><strong>Interpretation.</strong> ESGDI and ESG-NSI both rise slowly from 2023 to 2026 (median ESGDI about 0.50 to 0.53; median NSI about 0.11 to 0.16), consistent with firms expanding and polishing their reports as BRSR matured. The spread is wide enough to compare low and high firms.</p>



<h3 id="8.5-Is-the-BRSR-variable-informative?-Checks-on-the-moderator">8.5 Is the BRSR variable informative? Checks on the moderator</h3>


In [21]:
m = ['esgdi', 'nsi']
total_var = B[m].var()
between_var = B.groupby('firm')[m].transform('mean').var()          # variance of firm averages
print('Share of variance that is BETWEEN firms (the rest is within-firm change over time):')
print((between_var / total_var).round(3).to_string())
print()
print('Correlation matrix of the BRSR measures:')
B[['esgdi', 'nsi', 'net_sent', 'overall_sent', 'env', 'soc', 'gov']].corr().round(2)


Share of variance that is BETWEEN firms (the rest is within-firm change over time):
esgdi    0.684
nsi      0.671

Correlation matrix of the BRSR measures:


,esgdi,nsi,net_sent,overall_sent,env,soc,gov
esgdi,1.00,0.56,0.62,0.62,0.69,0.76,0.51
nsi,0.56,1.00,0.96,0.96,0.35,0.50,0.34
net_sent,0.62,0.96,1.00,1.00,0.39,0.54,0.36
overall_sent,0.62,0.96,1.00,1.00,0.39,0.54,0.37
env,0.69,0.35,0.39,0.39,1.00,0.15,0.20
soc,0.76,0.50,0.54,0.54,0.15,1.00,0.53
gov,0.51,0.34,0.36,0.37,0.20,0.53,1.00



<p><strong>Interpretation.</strong> About two-thirds of the variation in ESGDI (68%) and ESG-NSI (67%) is <strong>between firms</strong>; only a third is change over time. Fixed-effects interactions rely on that smaller within-firm part, so low power is expected, and a time-invariant version is added in robustness. ESGDI and ESG-NSI are moderately correlated (0.56), so they can enter together. ESG-NSI and the net sentiment index are almost identical (0.96), so the latter is not an independent check.</p>


In [22]:
print('Mean of env + soc + gov disclosure shares:', round((B.env + B.soc + B.gov).mean(), 3))
print('Mean ESGDI:', round(B.esgdi.mean(), 3), '-> pillar shares overlap, so they must not be added together')
within_change = B.groupby('firm').esgdi.agg(lambda s: s.max() - s.min()).describe().round(3)
print()
print('Within-firm range of ESGDI over 2023-26:')
print(within_change.to_string())


Mean of env + soc + gov disclosure shares: 0.651
Mean ESGDI: 0.514 -> pillar shares overlap, so they must not be added together

Within-firm range of ESGDI over 2023-26:
count    330.000
mean       0.047
std        0.035
min        0.000
25%        0.022
50%        0.043
75%        0.065
max        0.185



<p><strong>Interpretation.</strong> The three pillar shares add to 0.65, well above ESGDI (0.51), so pillars overlap and must not be summed. Within a firm, ESGDI moves by about 0.047 over four years, roughly one cross-firm standard deviation.</p>



<h3 id="8.6-Non-linearity-preview:-working-capital-deciles-versus-performance">8.6 Non-linearity preview: working capital deciles versus performance</h3>


In [23]:
A2 = A.copy()
A2['decile'] = pd.qcut(A2.lnCCC, 10, labels=False)
g = A2.groupby('decile').agg(lnCCC=('lnCCC', 'mean'), ROA=('ROA', 'mean'), se=('ROA', 'sem'))
plt.errorbar(g.lnCCC, g.ROA, yerr=1.96 * g.se, marker='o', capsize=3)
plt.xlabel('lnCCC (decile mean)  -  higher = longer cash cycle'); plt.ylabel('Mean ROA (%)')
plt.title('ROA across lnCCC deciles'); plt.show()



<p><strong>Interpretation.</strong> Mean ROA rises from about 5.7% in the shortest-cycle decile to about 9% in the third decile, stays near 8.5% through the middle, and falls to about 5.3% in the longest-cycle decile. This hump shape suggests a non-linear relationship, though it also mirrors the two clusters seen earlier.</p>



<h3 id="8.7-Correlation-and-multicollinearity">8.7 Correlation and multicollinearity</h3>


In [24]:
cv = ['lnCCC', 'ROA', 'ROE', 'Q', 'FirmSize', 'Leverage', 'Tangibility', 'Promoter', 'BoardInd', 'BoardSize', 'RnD']
cm = A[cv].corr()
fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(cm, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(cv))); ax.set_xticklabels(cv, rotation=60, ha='right')
ax.set_yticks(range(len(cv))); ax.set_yticklabels(cv); ax.grid(False)
for i in range(len(cv)):
    for j in range(len(cv)):
        ax.text(j, i, f'{cm.iloc[i, j]:.2f}', ha='center', va='center', fontsize=7)
plt.colorbar(im); plt.title('Correlation matrix (Sample A)'); plt.show()



<p><strong>Interpretation.</strong> <code>lnCCC</code> has almost no raw correlation with ROA (−0.01), ROE (−0.04) or Q (−0.05). Firm size is the strongest correlate of lnCCC (−0.20). ROA, ROE and Q are positively related (ROA–ROE 0.79, ROA–Q 0.46), as expected.</p>


In [25]:
X = A[['lnCCC', 'FirmSize', 'Leverage', 'Tangibility', 'Promoter', 'BoardInd', 'BoardSize', 'RnD']].dropna()
vif = {}
for c in X.columns:
    Z = np.column_stack([np.ones(len(X)), X.drop(columns=c)])
    resid = X[c] - Z @ np.linalg.lstsq(Z, X[c], rcond=None)[0]
    vif[c] = 1 / (resid.var() / X[c].var())            # VIF = 1 / (1 - R^2)
pd.Series(vif, name='VIF').round(2)


lnCCC          1.07
FirmSize       1.30
Leverage       1.14
Tangibility    1.22
Promoter       1.21
BoardInd       1.18
BoardSize      1.21
RnD            1.03
Name: VIF, dtype: float64


<p><strong>Interpretation.</strong> All variance inflation factors are between 1.0 and 1.3, far below the usual warning level of 5. Multicollinearity is not a concern.</p>



<h3 id="8.8-Selection-check:-do-firms-with-BRSR-scores-differ-from-those-without?">8.8 Selection check: do firms with BRSR scores differ from those without?</h3>


In [26]:
post = df[df.Post == 1].copy()
post['covered'] = post.groupby('firm').esgdi.transform('count') > 0
post.groupby('covered')[['FirmSize', 'ROA', 'ROE', 'Q', 'Leverage', 'lnCCC']].mean().round(3)


,FirmSize,ROA,ROE,Q,Leverage,lnCCC
covered,,,,,,
False,7.985,5.158,10.508,2.816,0.149,-1.243
True,9.286,8.583,15.828,3.915,0.111,-1.958



<p><strong>Interpretation.</strong> Firms with BRSR scores are larger (log assets 9.3 against 8.0), more profitable (ROA 8.6% against 5.2%) and have higher Tobin's Q (3.9 against 2.8). The BRSR results therefore speak for larger, better-reporting firms.</p>



<h2 id="9.-Regression-toolkit">9. Regression toolkit</h2>


In [27]:
def fe_ols(data, y, xs, fe=True, year_fe=True):
    d = data.dropna(subset=[y] + xs).copy()
    d = d[d.groupby('firm')[y].transform('size') > 1]
    X = d[xs].astype(float)
    if year_fe:
        X = pd.concat([X, pd.get_dummies(d.year, prefix='yr', drop_first=True, dtype=float)], axis=1)
    Y = d[y].astype(float)
    if fe:                                              # within transformation = firm fixed effects
        X = X - X.groupby(d.firm).transform('mean')
        Y = Y - Y.groupby(d.firm).transform('mean')
    else:
        X.insert(0, 'const', 1.0)
    XtX_inv = np.linalg.pinv(X.values.T @ X.values)
    b = XtX_inv @ X.values.T @ Y.values
    e = Y.values - X.values @ b
    S = pd.DataFrame(X.values * e[:, None]).groupby(d.firm.values).sum().values   # cluster scores
    G, N, K = S.shape[0], len(d), X.shape[1]
    V = G / (G - 1) * (N - 1) / (N - K) * XtX_inv @ (S.T @ S) @ XtX_inv          # cluster-robust variance
    se = np.sqrt(np.diag(V)); t = b / se; p = 2 * stats.t.sf(np.abs(t), G - 1)
    out = pd.DataFrame({'coef': b, 'se': se, 't': t, 'p': p}, index=X.columns)
    out['sig'] = np.where(p < .01, '***', np.where(p < .05, '**', np.where(p < .1, '*', '')))
    out.attrs.update(N=N, G=G, V=pd.DataFrame(V, X.columns, X.columns))
    return out

ctrl = ['FirmSize', 'Leverage', 'Tangibility', 'Promoter', 'BoardInd', 'BoardSize', 'RnD', 'RnD_rep']
test = fe_ols(df, 'ROA', ['lnCCC_c'] + ctrl)
print('Function check -> N =', test.attrs['N'], '| firms =', test.attrs['G'])


Function check -> N = 2601 | firms = 359



<p><strong>Interpretation.</strong> The function works: the ROA fixed-effects model uses 2,601 firm-years from 359 firms.</p>



<p><strong>Control variables.</strong> Firm age rises by exactly one each year, so with firm and year fixed effects it is perfectly collinear and is dropped from fixed-effects models (it is kept in pooled OLS). Inflation, GDP growth and the COVID dummy are likewise absorbed by year fixed effects and enter only the robustness model without year effects. P/B is excluded from the main models because it is missing for 17–30% of observations and overlaps with Tobin's Q; it is added in robustness.</p>



<h2 id="10.-Baseline:-working-capital-and-firm-performance-(H1)">10. Baseline: working capital and firm performance (H1)</h2>



<h3 id="10.1-Pooled-OLS-versus-firm-and-year-fixed-effects">10.1 Pooled OLS versus firm and year fixed effects</h3>


In [28]:
rows = []
for y in ['ROA', 'ROE', 'Q']:
    for name, kw, xs in [('Pooled OLS', dict(fe=False, year_fe=True), ['lnCCC_c'] + ctrl + ['lnAge']),
                         ('Firm + year FE', dict(fe=True, year_fe=True), ['lnCCC_c'] + ctrl)]:
        r = fe_ols(df, y, xs, **kw)
        rows.append({'outcome': y, 'model': name, 'coef_lnCCC': r.loc['lnCCC_c', 'coef'], 'se': r.loc['lnCCC_c', 'se'],
                     'p': r.loc['lnCCC_c', 'p'], 'sig': r.loc['lnCCC_c', 'sig'], 'N': r.attrs['N'], 'firms': r.attrs['G']})
pd.DataFrame(rows).round(4)


,outcome,model,coef_lnCCC,se,p,sig,N,firms
0,ROA,Pooled OLS,0.0438,0.0549,0.4262,,2601,359
1,ROA,Firm + year FE,-0.0255,0.0516,0.6205,,2601,359
2,ROE,Pooled OLS,-0.1363,0.1257,0.2789,,2567,358
3,ROE,Firm + year FE,-0.0992,0.1060,0.3496,,2567,358
4,Q,Pooled OLS,-0.0468,0.0271,0.0857,*,2601,359
5,Q,Firm + year FE,-0.0010,0.0225,0.9633,,2601,359



<p><strong>Interpretation.</strong> Once firm and year fixed effects are included, the lnCCC coefficient is close to zero and insignificant for all three outcomes (ROA −0.026, p = 0.62; ROE −0.099, p = 0.35; Q −0.001, p = 0.96). Only pooled OLS on Q shows a weak negative sign (p = 0.086). <strong>H1 is not supported on the full sample.</strong></p>



<h3 id="10.2-Full-fixed-effects-table-for-ROA-(primary-outcome)">10.2 Full fixed-effects table for ROA (primary outcome)</h3>


In [29]:
m1 = fe_ols(df, 'ROA', ['lnCCC_c'] + ctrl)
m1.loc[['lnCCC_c'] + ctrl].round(4)


,coef,se,t,p,sig
lnCCC_c,-0.0255,0.0516,-0.4956,0.6205,
FirmSize,-0.9519,0.7703,-1.2359,0.2173,
Leverage,-17.9862,2.8498,-6.3113,0.0000,***
Tangibility,-2.5244,1.0478,-2.4093,0.0165,**
Promoter,-0.0103,0.0237,-0.4348,0.6640,
BoardInd,0.0688,1.1013,0.0625,0.9502,
BoardSize,-0.0668,0.0640,-1.0443,0.2970,
RnD,-0.6648,0.2633,-2.5254,0.0120,**
RnD_rep,1.6807,0.6277,2.6777,0.0078,***



<p><strong>Interpretation.</strong> Among the controls, higher leverage lowers ROA strongly (a 0.1 rise in leverage lowers ROA by about 1.8 points, p &lt; 0.01), higher tangibility lowers it, and reporting R&amp;D raises it. Governance variables are insignificant, as expected because they barely change within firms.</p>



<h3 id="10.3-Robustness-to-the-bimodal-lnCCC">10.3 Robustness to the bimodal lnCCC</h3><p>Because <code>lnCCC</code> has two clusters, we repeat the baseline with:</p>
<p>(a) a scale-free measure, the within-year percentile rank of lnCCC.</p>
<p>(b) each cluster on its own.</p>


In [30]:
df['wcm_rank'] = df.groupby('year').lnCCC.rank(pct=True)           # 0 = shortest cycle, 1 = longest in that year
df['regime_hi'] = (df.lnCCC > 0).astype(int)

rows = []
for label, data, x in [('lnCCC, all firm-years', df, 'lnCCC_c'),
                       ('Percentile rank of lnCCC', df, 'wcm_rank'),
                       ('lnCCC > 0 cluster only', df[df.lnCCC > 0], 'lnCCC_c'),
                       ('lnCCC < 0 cluster only', df[df.lnCCC < 0], 'lnCCC_c')]:
    for y in ['ROA', 'ROE', 'Q']:
        r = fe_ols(data, y, [x] + ctrl)
        rows.append({'WCM measure': label, 'outcome': y, 'coef': r.loc[x, 'coef'], 'p': r.loc[x, 'p'],
                     'sig': r.loc[x, 'sig'], 'N': r.attrs['N']})
pd.DataFrame(rows).pivot(index='WCM measure', columns='outcome', values=['coef', 'p']).round(4)


coef 
 p 
 
 
 outcome 
 Q 
 ROA 
 ROE 
 Q 
 ROA 
 ROE 
 
 
 WCM measure 
 
 
 
 
 
 
 
 
 
 
 Percentile rank of lnCCC 
 0.2435 
 0.4457 
 0.6627 
 0.4880 
 0.5799 
 0.7141 
 
 
 lnCCC < 0 cluster only 
 0.1518 
 0.4786 
 0.6556 
 0.0606 
 0.0090 
 0.0809 
 
 
 lnCCC > 0 cluster only 
 -0.0234 
 -0.7459 
 -1.5269 
 0.8102 
 0.0031 
 0.0096 
 
 
 lnCCC, all firm-years 
 -0.0010 
 -0.0255 
 -0.0992 
 0.9633 
 0.6205 
 0.3496


<p><strong>Interpretation.</strong> The pooled null hides two opposite effects. In the <strong>upper cluster</strong> (lnCCC above 0) a longer cycle lowers ROA by 0.75 points per unit (p = 0.003) and ROE by 1.5 points (p = 0.010). In the <strong>lower cluster</strong> the slope is positive (ROA +0.48, p = 0.009). Opposite signs cancel in the full sample. The rank measure, which ignores scale, is insignificant. The upper-cluster result matches most of the literature (longer cycles hurt profitability); the lower-cluster result depends on how those values were computed.</p>



<h2 id="11.-Is-there-an-optimal-cash-conversion-cycle?">11. Is there an optimal cash conversion cycle?</h2><p>We add the square of centred lnCCC. An inverted U needs a negative squared term, a turning point inside the observed range, and a positive slope at low CCC and a negative slope at high CCC.</p>


In [31]:
rows = []
lo, hi = df.lnCCC_c.quantile([0.05, 0.95])
for y in ['ROA', 'ROE', 'Q']:
    r = fe_ols(df, y, ['lnCCC_c', 'lnCCC_sq'] + ctrl)
    b1, b2 = r.loc['lnCCC_c', 'coef'], r.loc['lnCCC_sq', 'coef']
    V = r.attrs['V']
    def slope(x):                                         # slope at x = b1 + 2*b2*x, with its standard error
        se = np.sqrt(V.loc['lnCCC_c', 'lnCCC_c'] + 4 * x**2 * V.loc['lnCCC_sq', 'lnCCC_sq'] + 4 * x * V.loc['lnCCC_c', 'lnCCC_sq'])
        return b1 + 2 * b2 * x, se
    s_lo, se_lo = slope(lo); s_hi, se_hi = slope(hi)
    tp = -b1 / (2 * b2) + df.lnCCC.mean()                 # turning point back on the lnCCC scale
    rows.append({'outcome': y, 'b_lnCCC': b1, 'b_sq': b2, 'p_sq': r.loc['lnCCC_sq', 'p'], 'turning_pt': tp,
                 'inside_5-95pct': bool(df.lnCCC.quantile(.05) < tp < df.lnCCC.quantile(.95)),
                 'slope_low': s_lo, 't_low': s_lo / se_lo, 'slope_high': s_hi, 't_high': s_hi / se_hi})
pd.DataFrame(rows).round(4)


,outcome,b_lnCCC,b_sq,p_sq,turning_pt,inside_5-95pct,slope_low,t_low,slope_high,t_high
0,ROA,0.0873,-0.0473,0.0006,-0.7564,True,0.5272,2.9878,-0.6050,-3.6201
1,ROE,0.0835,-0.0766,0.0157,-1.1338,True,0.7958,1.9752,-1.0373,-2.7266
2,Q,0.0193,-0.0085,0.1908,-0.5472,True,0.0984,1.2007,-0.1053,-1.3195



<p><strong>Interpretation.</strong> The squared term is negative and significant for ROA (−0.047, p &lt; 0.001) and ROE (−0.077, p = 0.016), but not for Q (p = 0.19). For ROA and ROE the slope is significantly positive at the 5th percentile of lnCCC and significantly negative at the 95th, and the turning point (about −0.8 to −1.1) lies inside the data range, so the inverted-U conditions are met. <strong>This hypothesis is supported for ROA and ROE.</strong>
One caution: the turning point sits in the thinly populated gap between the two clusters, so it is better read as 'firms in the lower cluster gain from longer cycles, firms in the upper cluster lose' than as a smooth optimum.</p>


In [32]:
r = fe_ols(df, 'ROA', ['lnCCC_c', 'lnCCC_sq'] + ctrl)
grid = np.linspace(lo, hi, 100)
fit = r.loc['lnCCC_c', 'coef'] * grid + r.loc['lnCCC_sq', 'coef'] * grid**2
plt.plot(grid + df.lnCCC.mean(), fit - fit.max())
plt.xlabel('lnCCC'); plt.ylabel('Predicted ROA relative to its peak (pp)')
plt.title('Fixed-effects fit of ROA on lnCCC (5th-95th percentile)'); plt.show()



<p><strong>Interpretation.</strong> The fitted curve peaks near lnCCC −0.8. At the extremes ROA is about 1.5 to 2 points below the peak.</p>



<h2 id="12.-Pre--versus-post-mandate-change-in-the-working-capital-effect-(part-1)">12. Pre- versus post-mandate change in the working-capital effect (part 1)</h2>


In [33]:
df['wcm_post'] = df.lnCCC_c * df.Post
rows = []
for y in ['ROA', 'ROE', 'Q']:
    r = fe_ols(df, y, ['lnCCC_c', 'wcm_post'] + ctrl)
    rows.append({'outcome': y, 'slope_pre': r.loc['lnCCC_c', 'coef'], 'p_pre': r.loc['lnCCC_c', 'p'],
                 'change_post': r.loc['wcm_post', 'coef'], 'se_change': r.loc['wcm_post', 'se'],
                 'p_change': r.loc['wcm_post', 'p'], 'sig': r.loc['wcm_post', 'sig'], 'N': r.attrs['N']})
pd.DataFrame(rows).round(4)


,outcome,slope_pre,p_pre,change_post,se_change,p_change,sig,N
0,ROA,-0.0188,0.7471,-0.0130,0.0524,0.8046,,2601
1,ROE,0.0622,0.6321,-0.3087,0.1253,0.0142,**,2567
2,Q,-0.0337,0.2191,0.0623,0.0252,0.0139,**,2601



<p><strong>Interpretation.</strong> The working-capital slope changed after 2022 for ROE (−0.31 per lnCCC unit, p = 0.014) and for Tobin's Q (+0.062, p = 0.014), but the two shifts point in opposite directions and ROA shows no change (p = 0.80). The evidence for a change in the relationship is mixed. This is a before/after association, not a mandate effect.</p>



<h2 id="13.-BRSR-as-a-moderator-of-the-working-capital-effect">13. BRSR as a moderator of the working-capital effect</h2><p>Only the post-mandate years (2023–2026) have BRSR scores, so these models use <code>df[df.Post == 1]</code>. BRSR measures are standardised, so the interaction reads: <em>change in the lnCCC slope for a one-standard-deviation higher BRSR score</em>. ESGDI (how much is disclosed) and ESG-NSI (how positive the tone is) enter in separate models.</p>


In [34]:
mods = ['esgdi_z', 'nsi_z', 'net_sent_z', 'env_z', 'soc_z', 'gov_z', 'esgdi_avg_z', 'nsi_avg_z']
for m in mods:
    df['wcm_' + m] = df.lnCCC_c * df[m]

post = df[df.Post == 1]
rows = []
for y in ['ROA', 'ROE', 'Q']:
    for m in ['esgdi_z', 'nsi_z']:
        r = fe_ols(post, y, ['lnCCC_c', m, 'wcm_' + m] + ctrl)
        rows.append({'outcome': y, 'moderator': m, 'b_lnCCC': r.loc['lnCCC_c', 'coef'], 'b_moderator': r.loc[m, 'coef'],
                     'b_interaction': r.loc['wcm_' + m, 'coef'], 'se': r.loc['wcm_' + m, 'se'],
                     'p_interaction': r.loc['wcm_' + m, 'p'], 'sig': r.loc['wcm_' + m, 'sig'], 'N': r.attrs['N'], 'firms': r.attrs['G']})
mod_table = pd.DataFrame(rows)
mod_table.round(4)


,outcome,moderator,b_lnCCC,b_moderator,b_interaction,se,p_interaction,sig,N,firms
0,ROA,esgdi_z,0.0532,0.1687,-0.0090,0.0429,0.8329,,1013,301
1,ROA,nsi_z,0.0564,0.1289,-0.0399,0.0450,0.3758,,1013,301
2,ROE,esgdi_z,-0.1796,0.4973,0.0743,0.1433,0.6046,,1003,298
3,ROE,nsi_z,-0.1704,0.5108,0.0440,0.1209,0.7163,,1003,298
4,Q,esgdi_z,-0.0208,0.0412,-0.0033,0.0160,0.8354,,1013,301
5,Q,nsi_z,-0.0203,-0.0110,-0.0180,0.0156,0.2490,,1013,301



<p><strong>Interpretation.</strong> None of the six interactions is significant (p between 0.25 and 0.84), and none survives the Holm adjustment. Precisely: on the pooled sample, firms with more extensive or more positive BRSR reporting do not show a different link between working capital and performance. <strong>These Hypothesis are not supported on the pooled sample.</strong></p>



<h3 id="13.1-Multiple-testing-adjustment">13.1 Multiple-testing adjustment</h3><p>Six primary tests (three outcomes by two moderators) are run, so we also report Holm-adjusted p-values.</p>


In [35]:
p = mod_table.p_interaction.values
order = np.argsort(p)
holm = np.empty(len(p))
holm[order] = np.minimum(1, np.maximum.accumulate((len(p) - np.arange(len(p))) * p[order]))   # Holm step-down
mod_table['p_holm'] = holm
mod_table[['outcome', 'moderator', 'b_interaction', 'p_interaction', 'p_holm']].round(4)


,outcome,moderator,b_interaction,p_interaction,p_holm
0,ROA,esgdi_z,-0.0090,0.8329,1.0
1,ROA,nsi_z,-0.0399,0.3758,1.0
2,ROE,esgdi_z,0.0743,0.6046,1.0
3,ROE,nsi_z,0.0440,0.7163,1.0
4,Q,esgdi_z,-0.0033,0.8354,1.0
5,Q,nsi_z,-0.0180,0.2490,1.0



<p><strong>Interpretation.</strong> After adjusting for six tests the smallest p-value is 0.25 (unadjusted), so every adjusted p is 1.0. The null result is not caused by the adjustment.</p>



<h3 id="13.2-What-does-the-interaction-mean?-Marginal-effects-of-lnCCC">13.2 What does the interaction mean? Marginal effects of lnCCC</h3>


In [36]:
fig, ax = plt.subplots(1, 3, figsize=(13, 4), sharey=False)
for a, y in zip(ax, ['ROA', 'ROE', 'Q']):
    for k, (m, lab) in enumerate([('esgdi_z', 'ESGDI'), ('nsi_z', 'ESG-NSI')]):
        r = fe_ols(post, y, ['lnCCC_c', m, 'wcm_' + m] + ctrl); V = r.attrs['V']
        zs = post[m].quantile([0.1, 0.5, 0.9]).values
        slope = r.loc['lnCCC_c', 'coef'] + r.loc['wcm_' + m, 'coef'] * zs
        se = np.sqrt(V.loc['lnCCC_c', 'lnCCC_c'] + zs**2 * V.loc['wcm_' + m, 'wcm_' + m] + 2 * zs * V.loc['lnCCC_c', 'wcm_' + m])
        a.errorbar(np.arange(3) + 0.12 * k, slope, yerr=1.96 * se, marker='o', capsize=4, label=lab)
    a.axhline(0, color='k', lw=0.8); a.set_xticks(range(3)); a.set_xticklabels(['10th pct', 'median', '90th pct'])
    a.set_title(f'Effect of lnCCC on {y}'); a.legend()
plt.suptitle('Slope of lnCCC at low / median / high BRSR score (95% CI)'); plt.tight_layout(); plt.show()



<p><strong>Interpretation.</strong> At low, median and high BRSR the lnCCC slope stays close to zero and the confidence intervals overlap heavily for all three outcomes. The lines are nearly flat, which is what 'no moderation' looks like.</p>



<h3 id="13.3-Does-tone-matter-beyond-disclosure-volume?-Combined-and-abnormal-tone-models">13.3 Does tone matter beyond disclosure volume? Combined and abnormal-tone models</h3><p>To claim that tone matters <em>independently</em> of how much is disclosed, both moderators must be in the same model. We also build an <strong>abnormal tone</strong> variable: the part of ESG-NSI not explained by size, leverage, tangibility, profitability and year. It separates narrative spin from fundamentals.</p>


In [37]:
print('Correlation between ESGDI and ESG-NSI (Sample B):', round(post[['esgdi', 'nsi']].corr().iloc[0, 1], 3))

# abnormal tone = residual of NSI regressed on fundamentals and year dummies
Bd = post.dropna(subset=['nsi', 'FirmSize', 'Leverage', 'Tangibility', 'ROA'])
Z = np.column_stack([np.ones(len(Bd)), Bd[['FirmSize', 'Leverage', 'Tangibility', 'ROA']],
                     pd.get_dummies(Bd.year, drop_first=True, dtype=float)])
resid = Bd.nsi - Z @ np.linalg.lstsq(Z, Bd.nsi, rcond=None)[0]
df['nsi_abn_z'] = (resid - resid.mean()) / resid.std()
df['wcm_nsi_abn_z'] = df.lnCCC_c * df.nsi_abn_z
post = df[df.Post == 1]

rows = []
for y in ['ROA', 'ROE', 'Q']:
    r = fe_ols(post, y, ['lnCCC_c', 'esgdi_z', 'nsi_z', 'wcm_esgdi_z', 'wcm_nsi_z'] + ctrl)
    r2 = fe_ols(post, y, ['lnCCC_c', 'nsi_abn_z', 'wcm_nsi_abn_z'] + ctrl)
    rows.append({'outcome': y, 'ESGDI x lnCCC (combined)': r.loc['wcm_esgdi_z', 'coef'], 'p_1': r.loc['wcm_esgdi_z', 'p'],
                 'NSI x lnCCC (combined)': r.loc['wcm_nsi_z', 'coef'], 'p_2': r.loc['wcm_nsi_z', 'p'],
                 'Abnormal-tone x lnCCC': r2.loc['wcm_nsi_abn_z', 'coef'], 'p_3': r2.loc['wcm_nsi_abn_z', 'p']})
pd.DataFrame(rows).round(4)


Correlation between ESGDI and ESG-NSI (Sample B): 0.557


,outcome,ESGDI x lnCCC (combined),p_1,NSI x lnCCC (combined),p_2,Abnormal-tone x lnCCC,p_3
0,ROA,0.0226,0.5918,-0.0516,0.2889,-0.0187,0.6591
1,ROE,0.0640,0.6229,0.0114,0.9146,0.0766,0.5053
2,Q,0.0122,0.5037,-0.0243,0.1711,-0.0176,0.2863



<p><strong>Interpretation.</strong> With both measures in one model, and with the abnormal-tone version, no interaction is significant (all p above 0.17). Tone therefore does not matter independently of disclosure volume in this sample. <strong>Hypothesis is not supported.</strong></p>



<h3 id="13.4-Which-part-of-BRSR-matters?-Environmental,-social-and-governance-disclosure-separately">13.4 Which part of BRSR matters? Environmental, social and governance disclosure separately</h3><p>The three pillar shares overlap with each other, so they are entered one at a time and never summed.</p>


In [38]:
rows = []
for y in ['ROA', 'ROE', 'Q']:
    for m in ['env_z', 'soc_z', 'gov_z']:
        r = fe_ols(post, y, ['lnCCC_c', m, 'wcm_' + m] + ctrl)
        rows.append({'outcome': y, 'pillar': m, 'b_interaction': r.loc['wcm_' + m, 'coef'], 'p': r.loc['wcm_' + m, 'p'], 'sig': r.loc['wcm_' + m, 'sig']})
pd.DataFrame(rows).pivot(index='pillar', columns='outcome', values=['b_interaction', 'p']).round(4)


b_interaction 
 p 
 
 
 outcome 
 Q 
 ROA 
 ROE 
 Q 
 ROA 
 ROE 
 
 
 pillar 
 
 
 
 
 
 
 
 
 
 
 env_z 
 -0.0025 
 -0.0652 
 0.0212 
 0.8732 
 0.1102 
 0.9040 
 
 
 gov_z 
 -0.0043 
 -0.0128 
 0.0836 
 0.7493 
 0.7639 
 0.6417 
 
 
 soc_z 
 -0.0127 
 0.0646 
 0.1452 
 0.4451 
 0.1003 
 0.1858


<p><strong>Interpretation.</strong> No pillar shows a significant interaction at 5%. The closest are environmental disclosure on ROA (−0.065, p = 0.11) and social disclosure on ROA (+0.065, p = 0.10), pointing in opposite directions.</p>



<h2 id="14.-Does-BRSR-change-the-optimal-cycle?">14. Does BRSR change the optimal cycle?</h2><p>We interact BRSR with both lnCCC and its square, then compute the optimum at low, median and high BRSR. An optimum exists only if the quadratic term at that BRSR level is negative.</p>


In [40]:
df['sq_x_esgdi'] = df.lnCCC_sq * df.esgdi_z
df['sq_x_nsi'] = df.lnCCC_sq * df.nsi_z
post = df[df.Post == 1]
rows = []
for m, sq in [('esgdi_z', 'sq_x_esgdi'), ('nsi_z', 'sq_x_nsi')]:
    r = fe_ols(post, 'ROA', ['lnCCC_c', 'lnCCC_sq', m, 'wcm_' + m, sq] + ctrl)
    c = r.coef
    for q in [0.1, 0.5, 0.9]:
        z = post[m].quantile(q)
        lin, quad = c['lnCCC_c'] + c['wcm_' + m] * z, c['lnCCC_sq'] + c[sq] * z
        rows.append({'moderator': m, 'BRSR level': f'{int(q*100)}th pct', 'linear': lin, 'quadratic': quad,
                     'optimum_lnCCC': (-lin / (2 * quad) + df.lnCCC.mean()) if quad < 0 else np.nan,
                     'p_interaction_sq': r.loc[sq, 'p']})
pd.DataFrame(rows).round(4)


,moderator,BRSR level,linear,quadratic,optimum_lnCCC,p_interaction_sq
0,esgdi_z,10th pct,0.2596,-0.1065,-0.4602,0.1441
1,esgdi_z,50th pct,0.1807,-0.0763,-0.4942,0.1441
2,esgdi_z,90th pct,0.1097,-0.0491,-0.5607,0.1441
3,nsi_z,10th pct,0.3286,-0.1121,-0.2123,0.1150
4,nsi_z,50th pct,0.2028,-0.0799,-0.4105,0.1150
5,nsi_z,90th pct,0.0545,-0.0421,-1.0319,0.1150



<p><strong>Interpretation.</strong> The interaction of BRSR with the squared term is insignificant (p = 0.14 for ESGDI, 0.12 for ESG-NSI). The implied optimum barely moves with ESGDI (−0.46 to −0.56). With ESG-NSI it moves from −0.21 to −1.03, but this shift is not statistically distinguishable from zero. <strong>This Hypothesis is not supported.</strong></p>



<h2 id="15.-Speed-of-adjustment-toward-a-target-cycle">15. Speed of adjustment toward a target cycle</h2><p>A firm's <em>target</em> is the median working-capital position of firms of similar size in the same year. The <em>gap</em> is the target minus last year's position. If firms close the gap, the change in position is positively related to the gap, and a positive interaction with BRSR means high-BRSR firms close it faster. Percentile ranks are used so the result does not depend on the scale of lnCCC.</p>


In [41]:
df['rank_lag'] = df.groupby('firm').wcm_rank.shift(1)
df['d_rank'] = df.wcm_rank - df.rank_lag
df['size_t'] = df.groupby('year').FirmSize.transform(lambda s: pd.qcut(s, 3, labels=False, duplicates='drop'))
df['target'] = df.groupby(['year', 'size_t']).wcm_rank.transform('median')
df['gap'] = df.target - df.rank_lag
for m in ['esgdi_z', 'nsi_z']:
    df['gap_' + m] = df.gap * df[m]
post = df[df.Post == 1]

rows = []
for m in ['esgdi_z', 'nsi_z']:
    r = fe_ols(post, 'd_rank', ['gap', m, 'gap_' + m] + ctrl)
    rows.append({'moderator': m, 'adjustment_speed': r.loc['gap', 'coef'], 'p_speed': r.loc['gap', 'p'],
                 'speed_x_BRSR': r.loc['gap_' + m, 'coef'], 'p_interaction': r.loc['gap_' + m, 'p'], 'N': r.attrs['N']})
pd.DataFrame(rows).round(4)


,moderator,adjustment_speed,p_speed,speed_x_BRSR,p_interaction,N
0,esgdi_z,0.597,0.0,-0.0240,0.7093,1002
1,nsi_z,0.591,0.0,0.0295,0.5941,1002



<p><strong>Interpretation.</strong> Firms close about 60% of the gap to their size-and-year peer median each year (p &lt; 0.001). Part of this is mechanical, because ranks tend to revert. The BRSR interactions (−0.024 and +0.030) are tiny and insignificant, so high-BRSR firms do not adjust faster. <strong>Hypothesis is not supported.</strong></p>



<h2 id="16.-Before-and-after-the-mandate,-by-BRSR-exposure-(part-2)">16. Before and after the mandate, by BRSR exposure (part 2)</h2><p>All Nifty 500 firms are covered by the mandate, so there is no untreated group. Instead we compare firms with <strong>high</strong> versus <strong>low</strong> average BRSR disclosure (median split of 2023–26 ESGDI). The coefficient on the triple interaction shows whether the working-capital slope changed more for high-BRSR firms after 2022. Because exposure is measured after the mandate, this is an association, not a causal estimate. The event-study plot checks that the slopes did not already diverge before 2023.</p>


In [43]:
E = df[df.esgdi_avg.notna()].copy()
E['high'] = (E.esgdi_avg > E.esgdi_avg.median()).astype(int)
E['w_high'] = E.lnCCC_c * E.high
E['w_post_high'] = E.lnCCC_c * E.Post * E.high

rows = []
for y in ['ROA', 'ROE', 'Q']:
    r = fe_ols(E, y, ['lnCCC_c', 'wcm_post', 'w_high', 'w_post_high'] + ctrl)
    rows.append({'outcome': y, 'slope_low_pre': r.loc['lnCCC_c', 'coef'], 'post_change_low': r.loc['wcm_post', 'coef'],
                 'high_vs_low_pre': r.loc['w_high', 'coef'], 'triple_interaction': r.loc['w_post_high', 'coef'],
                 'p_triple': r.loc['w_post_high', 'p'], 'sig': r.loc['w_post_high', 'sig'], 'N': r.attrs['N'], 'firms': r.attrs['G']})
pd.DataFrame(rows).round(4)


,outcome,slope_low_pre,post_change_low,high_vs_low_pre,triple_interaction,p_triple,sig,N,firms
0,ROA,-0.0751,0.0349,0.0987,-0.0632,0.5682,,2419,329
1,ROE,-0.0400,-0.3170,0.1541,0.0970,0.6810,,2387,328
2,Q,-0.0374,0.0190,0.0019,0.0796,0.1424,,2419,329



<p><strong>Interpretation.</strong> The triple interaction is insignificant for ROA (−0.063, p = 0.57), ROE (+0.097, p = 0.68) and Q (+0.080, p = 0.14). The working-capital slope did not change differently after 2022 for high-BRSR firms. <strong>Hypothesis (BRSR-exposure part) is not supported.</strong></p>


In [44]:
yrs = [2019, 2020, 2021, 2023, 2024, 2025, 2026]               # 2022 is the reference year
for y in yrs:
    E[f'w_{y}'] = E.lnCCC_c * (E.year == y)
    E[f'wh_{y}'] = E.w_high * (E.year == y)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, out in zip(ax, ['ROA', 'Q']):
    r = fe_ols(E, out, ['lnCCC_c', 'w_high'] + [f'w_{y}' for y in yrs] + [f'wh_{y}' for y in yrs] + ctrl)
    c = r.loc[[f'wh_{y}' for y in yrs]]
    xs = np.arange(len(yrs))
    a.errorbar(xs, c.coef, yerr=1.96 * c.se, fmt='o', capsize=4)
    a.axhline(0, color='k', lw=0.8); a.axvline(2.5, color='grey', ls=':')
    a.set_xticks(xs); a.set_xticklabels(yrs); a.set_title(f'{out}: lnCCC slope, high minus low BRSR firms (vs 2022)')
plt.tight_layout(); plt.show()



<p><strong>Interpretation.</strong> Before 2022 the high-minus-low slope differences are centred on zero with intervals that include zero, so no pre-trend appears. After 2022 they also stay around zero. The design is clean but shows no effect.</p>



<h2 id="17.-COVID-19-shock-test">17. COVID-19 shock test</h2><p>The pandemic was a shock firms did not choose. If efficient working capital protects performance, firms that entered the pandemic with a <em>longer</em> cycle should lose more performance in the COVID year. The firm's 2019 position is fixed in advance, so reverse causality is not a concern for this test.</p>


In [45]:
df['pre19'] = df.firm.map(df[df.year == 2019].set_index('firm').lnCCC)
df['pre19_c'] = df.pre19 - df.pre19.mean()
df['pre19_covid'] = df.pre19_c * df.CovidDummy
df['pre19_rank'] = df.firm.map(df[df.year == 2019].set_index('firm').wcm_rank)
df['pre19_rank_covid'] = (df.pre19_rank - 0.5) * df.CovidDummy

rows = []
for y in ['ROA', 'ROE', 'Q']:
    for lab, x in [('2019 lnCCC x COVID', 'pre19_covid'), ('2019 rank x COVID', 'pre19_rank_covid')]:
        r = fe_ols(df, y, [x] + ctrl)
        rows.append({'outcome': y, 'term': lab, 'coef': r.loc[x, 'coef'], 'se': r.loc[x, 'se'], 'p': r.loc[x, 'p'], 'sig': r.loc[x, 'sig'], 'N': r.attrs['N']})
pd.DataFrame(rows).round(4)


,outcome,term,coef,se,p,sig,N
0,ROA,2019 lnCCC x COVID,0.0311,0.0825,0.7064,,2443
1,ROA,2019 rank x COVID,-0.2139,1.3912,0.8779,,2443
2,ROE,2019 lnCCC x COVID,0.2622,0.1639,0.1107,,2424
3,ROE,2019 rank x COVID,3.0490,2.6377,0.2486,,2424
4,Q,2019 lnCCC x COVID,-0.0448,0.0240,0.0628,*,2443
5,Q,2019 rank x COVID,-0.9454,0.3848,0.0145,**,2443



<p><strong>Interpretation.</strong> For Tobin's Q, firms that entered 2020 with a longer cycle lost more value in the COVID year: the 2019 rank coefficient is −0.95 (p = 0.015) and the 2019 lnCCC coefficient is −0.045 (p = 0.063). Moving from the shortest to the longest cycle in 2019 is associated with a 0.95 lower Q in 2020. ROA and ROE show no significant effect. This is the only result that uses a shock firms did not choose, and it suggests markets penalised long cycles more than accounting profits did. It is a single result among many, so treat it as suggestive.</p>



<h2 id="18.-Endogeneity-checks">18. Endogeneity checks</h2><ol>
<li><strong>Lagged WCM:</strong> does last year's working capital predict this year's performance?</li>
<li><strong>Reverse direction:</strong> does last year's performance predict this year's working capital? If it does, simple contemporaneous regressions may reflect reverse causality.
Industry codes are not in the data, so an industry-mean instrument and System GMM are not used.</li>
</ol>


In [47]:
rows = []
for y in ['ROA', 'ROE', 'Q']:
    r = fe_ols(df, y, ['lnCCC_lag'] + ctrl)
    rows.append({'test': f'{y} on lagged lnCCC', 'coef': r.loc['lnCCC_lag', 'coef'], 'p': r.loc['lnCCC_lag', 'p'], 'N': r.attrs['N']})
for y_lag in ['ROA_lag']:
    r = fe_ols(df, 'lnCCC', [y_lag] + ctrl)
    rows.append({'test': 'lnCCC on lagged ROA', 'coef': r.loc[y_lag, 'coef'], 'p': r.loc[y_lag, 'p'], 'N': r.attrs['N']})
r = fe_ols(df, 'wcm_rank', ['ROA_lag'] + ctrl)
rows.append({'test': 'WCM rank on lagged ROA', 'coef': r.loc['ROA_lag', 'coef'], 'p': r.loc['ROA_lag', 'p'], 'N': r.attrs['N']})
pd.DataFrame(rows).round(4)


,test,coef,p,N
0,ROA on lagged lnCCC,0.0089,0.8748,2264
1,ROE on lagged lnCCC,0.0418,0.7182,2236
2,Q on lagged lnCCC,-0.0030,0.8845,2264
3,lnCCC on lagged ROA,-0.0050,0.7583,2292
4,WCM rank on lagged ROA,0.0001,0.9158,2292



<p><strong>Interpretation.</strong> Last year's lnCCC does not predict this year's ROA, ROE or Q (p between 0.72 and 0.88), and last year's ROA does not predict lnCCC (p = 0.76 and 0.92). There is no sign of reverse causality in this short panel, but the lack of any predictive link also matches the null baseline. Industry-mean instruments and System GMM need an industry variable and a longer panel, so they are not run.</p>



<h2 id="19.-Robustness-of-the-BRSR-moderation-result">19. Robustness of the BRSR moderation result</h2><p>The table repeats the ROA interaction under alternative timing, samples, controls and WCM measures. Each row reports the interaction coefficient (WCM × BRSR).</p>


In [48]:
df['esgdi_z_lag'] = df.groupby('firm').esgdi_z.shift(1)
df['nsi_z_lag'] = df.groupby('firm').nsi_z.shift(1)
post = df[df.Post == 1]
ctrl_pb = ctrl + ['PB']
macro = ['Infl', 'GDP']                      # COVID dummy is all zeros in 2023-26, so it is not needed here

specs = [  # label, data, WCM variable, controls, year FE?, moderator name pattern
    ('Baseline (2023-26, contemporaneous)', post, 'lnCCC_c', ctrl, True, '{}_z'),
    ('Moderator lagged one year', post, 'lnCCC_c', ctrl, True, '{}_z_lag'),
    ('Firm-average moderator (time-invariant)', post, 'lnCCC_c', ctrl, True, '{}_avg_z'),
    ('Excluding 2026 (carried-forward leverage)', post[post.year < 2026], 'lnCCC_c', ctrl, True, '{}_z'),
    ('Adding P/B ratio', post, 'lnCCC_c', ctrl_pb, True, '{}_z'),
    ('No year effects; macro controls instead', post[post.year < 2026], 'lnCCC_c', ctrl + macro, False, '{}_z'),
    ('WCM as percentile rank', post, 'wcm_rank', ctrl, True, '{}_z'),
    ('lnCCC > 0 cluster only', post[post.lnCCC > 0], 'lnCCC_c', ctrl, True, '{}_z'),
    ('lnCCC < 0 cluster only', post[post.lnCCC < 0], 'lnCCC_c', ctrl, True, '{}_z')]

rows = []
for label, data, w, controls, yfe, pat in specs:
    row = {'specification': label}
    for base in ['esgdi', 'nsi']:
        mcol = pat.format(base)
        d = data.assign(inter=data[w] * data[mcol])
        xs = [w, 'inter'] + ([] if 'avg' in pat else [mcol]) + controls      # firm-average moderator is absorbed by firm effects
        r = fe_ols(d, 'ROA', xs, year_fe=yfe)
        row[base + '_coef'], row[base + '_p'] = r.loc['inter', 'coef'], r.loc['inter', 'p']
    row['N'] = r.attrs['N']
    rows.append(row)
pd.DataFrame(rows).round(4)


,specification,esgdi_coef,esgdi_p,nsi_coef,nsi_p,N
0,"Baseline (2023-26, contemporaneous)",-0.0090,0.8329,-0.0399,0.3758,1013
1,Moderator lagged one year,0.0445,0.3137,0.0225,0.6086,690
2,Firm-average moderator (time-invariant),-0.1021,0.1259,-0.0909,0.0713,1211
3,Excluding 2026 (carried-forward leverage),-0.0152,0.7389,-0.0548,0.2704,734
4,Adding P/B ratio,-0.0104,0.8044,-0.0368,0.4106,1003
5,No year effects; macro controls instead,-0.0152,0.7389,-0.0548,0.2704,734
6,WCM as percentile rank,-0.0464,0.9458,-0.5216,0.4778,1013
7,lnCCC > 0 cluster only,0.6863,0.0352,0.4468,0.1468,283
8,lnCCC < 0 cluster only,-0.0682,0.5508,-0.1713,0.3725,686



<p><strong>Interpretation.</strong> The moderation result is stable: in eight of nine specifications neither interaction is significant at 5%. Two exceptions: the firm-average NSI interaction is borderline negative (−0.091, p = 0.071), and in the upper cluster the ESGDI interaction is positive and significant (+0.686, p = 0.035). The 'no year effects' row equals the 'excluding 2026' row because, over three years, inflation and GDP carry exactly the same information as year dummies.</p>



<h2 id="20.-Secondary-analysis:-firms-with-a-conventional-cash-cycle-(lnCCC-%3E-0)">20. Secondary analysis: firms with a conventional cash cycle (lnCCC &gt; 0)</h2><p>This section repeats the full set of BRSR tests on that cluster, with Holm adjustment across the six tests. Treat it as exploratory, because the sample is small.</p>


In [49]:
pos = post[post.lnCCC > 0]
rows = []
for y in ['ROA', 'ROE', 'Q']:
    for m in ['esgdi_z', 'nsi_z']:
        r = fe_ols(pos, y, ['lnCCC_c', m, 'wcm_' + m] + ctrl)
        rows.append({'outcome': y, 'moderator': m, 'b_lnCCC': r.loc['lnCCC_c', 'coef'], 'b_interaction': r.loc['wcm_' + m, 'coef'],
                     'se': r.loc['wcm_' + m, 'se'], 'p': r.loc['wcm_' + m, 'p'], 'N': r.attrs['N'], 'firms': r.attrs['G']})
up = pd.DataFrame(rows)
p = up.p.values; order = np.argsort(p); h = np.empty(len(p))
h[order] = np.minimum(1, np.maximum.accumulate((len(p) - np.arange(len(p))) * p[order]))
up['p_holm'] = h
up.round(4)


,outcome,moderator,b_lnCCC,b_interaction,se,p,N,firms,p_holm
0,ROA,esgdi_z,-1.4357,0.6863,0.3213,0.0352,283,97,0.2113
1,ROA,nsi_z,-1.3055,0.4468,0.3055,0.1468,283,97,0.7340
2,ROE,esgdi_z,-0.4967,0.1058,1.1265,0.9254,283,97,1.0000
3,ROE,nsi_z,-0.5041,-0.1862,0.8191,0.8206,283,97,1.0000
4,Q,esgdi_z,-0.0019,0.0041,0.0733,0.9554,283,97,1.0000
5,Q,nsi_z,0.0072,0.0545,0.0795,0.4952,283,97,1.0000



<p><strong>Interpretation.</strong> In the upper cluster (283 firm-years, 97 firms) a longer cycle lowers ROA by 1.4 points per unit for the average firm, and a one-SD higher ESGDI <strong>halves</strong> that penalty (interaction +0.69, p = 0.035). After Holm adjustment for six tests the p-value is 0.21, so it is <strong>not significant once multiple testing is considered</strong>. ROE and Q show nothing.</p>


In [50]:
fig, ax = plt.subplots(figsize=(6.5, 4))
for k, (m, lab) in enumerate([('esgdi_z', 'ESGDI'), ('nsi_z', 'ESG-NSI')]):
    r = fe_ols(pos, 'ROA', ['lnCCC_c', m, 'wcm_' + m] + ctrl); V = r.attrs['V']
    zs = pos[m].quantile([0.1, 0.5, 0.9]).values
    slope = r.loc['lnCCC_c', 'coef'] + r.loc['wcm_' + m, 'coef'] * zs
    se = np.sqrt(V.loc['lnCCC_c', 'lnCCC_c'] + zs**2 * V.loc['wcm_' + m, 'wcm_' + m] + 2 * zs * V.loc['lnCCC_c', 'wcm_' + m])
    ax.errorbar(np.arange(3) + 0.12 * k, slope, yerr=1.96 * se, marker='o', capsize=4, label=lab)
ax.axhline(0, color='k', lw=0.8); ax.set_xticks(range(3)); ax.set_xticklabels(['10th pct', 'median', '90th pct'])
ax.set_title('lnCCC > 0 cluster: effect of lnCCC on ROA by BRSR level'); ax.legend(); plt.show()



<p><strong>Interpretation.</strong> For long-cycle firms with low ESGDI (10th percentile) each extra unit of lnCCC costs about 2.3 ROA points (interval roughly −3.8 to −0.8). At the median it costs about 1.5, and at the 90th percentile about 0.55, with an interval that includes zero. This pattern matches the theory that more disclosure reduces the cost of long cycles, but it is exploratory and should be reported as such.</p>



<h2 id="21.-Summary-of-results">21. Summary of results</h2>



<h3 id="Hypothesis-scorecard">Hypothesis scorecard</h3><table>
<thead>
<tr>
<th>Hypothesis</th>
<th>Result</th>
<th>Evidence</th>
</tr>
</thead>
<tbody>
<tr>
<td>Longer CCC lowers performance</td>
<td><strong>Not supported</strong> on the full sample; supported within the lnCCC &gt; 0 cluster</td>
<td>FE coefficient ≈ 0 (p 0.35–0.96); upper cluster ROA −0.75 (p = 0.003)</td>
</tr>
<tr>
<td>Inverted-U with an optimum</td>
<td><strong>Supported</strong> for ROA and ROE; not for Q</td>
<td>Squared term −0.047 (p &lt; 0.001) for ROA; turning point lies in the gap between clusters</td>
</tr>
<tr>
<td>BRSR disclosure (ESGDI) moderates</td>
<td><strong>Not supported</strong> pooled; exploratory support in the upper cluster</td>
<td>Pooled p 0.60–0.84; upper cluster ROA +0.69 (p = 0.035, Holm 0.21)</td>
</tr>
<tr>
<td>BRSR tone (ESG-NSI) moderates beyond disclosure</td>
<td><strong>Not supported</strong></td>
<td>All p above 0.17 in combined and abnormal-tone models</td>
</tr>
<tr>
<td>BRSR shifts the optimal CCC</td>
<td><strong>Not supported</strong></td>
<td>Interaction with squared term p = 0.12–0.14</td>
</tr>
<tr>
<td>Relationship changes after the mandate, more for high-BRSR firms</td>
<td><strong>Mixed / not supported</strong></td>
<td>ROE and Q change in opposite directions; triple interaction p ≥ 0.14</td>
</tr>
<tr>
<td>High-BRSR firms adjust faster</td>
<td><strong>Not supported</strong></td>
<td>Interaction p 0.59–0.71</td>
</tr>
<tr>
<td>COVID test</td>
<td><strong>Suggestive</strong> for Tobin's Q</td>
<td>2019 rank × COVID −0.95 (p = 0.015)</td>
</tr>
</tbody>
</table>
<h3 id="What-this-means">What this means</h3><ol>
<li><strong>The honest headline is a null on the central question.</strong> On the pooled 2023–2026 sample, BRSR disclosure and sentiment do not change how working capital relates to performance. The result survives lags, alternative samples, controls and measures.</li>
<li><strong>Working capital matters, but not in a simple linear way.</strong> The inverted-U for ROA and ROE, the opposite slopes in the two lnCCC clusters and the COVID result on Tobin's Q all point to a non-linear, state-dependent relationship.</li>
<li><strong>The one signal for BRSR comes from long-cycle firms.</strong> There, disclosure appears to reduce the profit penalty of a long cycle. It does not survive multiple-testing correction, so it is a lead for further work, not a finding.</li>
<li><strong>Novel contributions delivered:</strong> the first test of BRSR disclosure and tone as moderators of working capital effects; an abnormal-tone measure; BRSR-dependent optima; adjustment-speed tests; an event-study design around the mandate; and a COVID shock test.</li>
</ol>
